# Train WavLM-Base + ASP on ASVspoof 2019 LA

This notebook implements a **lightweight logical-access (LA) anti-spoofing** model, following:

> Zhang et al., *A lightweight end-to-end anti-spoofing voice model based on WavLM*, ICACS 2024  
> https://doi.org/10.1145/3708597.3708621

## What the paper does

- **Problem:** detect **synthetic spoof** (TTS / voice conversion), not loudspeaker **replay**.
- **Frontend:** WavLM-Base self-supervised encoder (speech SSL).
- **Backend:** **attentive statistics pooling (ASP)** + a few fully connected layers.
- **Claim:** cheaper than heavy SOTA detectors, **0.45% EER** on **ASVspoof 2019 LA**.

## What we train here

| Piece | Choice |
|---|---|
| Encoder | `microsoft/wavlm-base` (frozen by default) |
| Pooling | ASP (learnable time weights → mean + std) |
| Head | 3-layer FC → 1 logit (bona fide vs spoof) |
| Data | ASVspoof 2019 **LA train**, speaker-disjoint val |

First run downloads WavLM from Hugging Face (~360MB).

**Do not compare this EER to replay (2017 / PA) numbers.** Different attack, different corpus.

## Environment check

Needs GPU if you want a full run. `data/LA` must exist (protocols + flac).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "train_lib.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\wavlm_la2019")
sys.path.insert(0, str(ROOT))

import torch
from experiment_lib import DEFAULT_LA, PAPER_LA_EER_PERCENT
from train_lib import train_wavlm_la

print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("LA:", DEFAULT_LA, "exists", DEFAULT_LA.exists())
print("Paper LA EER (reference):", PAPER_LA_EER_PERCENT, "%")

## Knobs

- `SMOKE = True` — small subset, 1 epoch (sanity + HF download).
- Then set `SMOKE = False` for a real train.
- If you OOM, drop `BATCH_SIZE` to `2`.
- `FREEZE_ENCODER = True` matches the paper’s cheap-backend idea. Unfreeze only if you have VRAM and time.

In [ ]:
SMOKE = True
FREEZE_ENCODER = True
EPOCHS = 1 if SMOKE else 12
PATIENCE = 2 if SMOKE else 4
BATCH_SIZE = 2 if SMOKE else 4
MAX_TRAIN = 200 if SMOKE else 0
MAX_VAL = 100 if SMOKE else 0
FORCE_CPU = False

## Train

Checkpoint → `runs/wavlm_la/best_wavlm_la2019.pt`  
Only **ASP + FC head** are saved; WavLM reloads from Hugging Face at eval time.

Val EER is on a **speaker-held-out slice of LA train**, not official LA eval. Official numbers come from notebook 02 (`SPLIT = "dev"` or `"eval"`).

In [ ]:
ckpt = train_wavlm_la(
    epochs=EPOCHS,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    max_train=MAX_TRAIN,
    max_val=MAX_VAL,
    freeze_encoder=FREEZE_ENCODER,
    force_cpu=FORCE_CPU,
)
ckpt

## After training

Open `02_eval_wavlm_la.ipynb` and score **LA dev**.  
Optional: compare oracle EER with `../lfcc_la2019/runs/` if that experiment exists.

If speaker-val EER is near chance (~50%), the smoke subset is too small — rerun with `SMOKE = False`.